### Your name:

<pre> Ju Zhang</pre>

### Collaborators:

<pre> Enter the name of the people you worked with if any</pre>


### Build full pipeline for the data analysis following the example of the notebook.
 Hint: the main part requested to change is the algorithm used (KNN regression)


#### Considerations for building pipeline:

- Make your notebook as compact as possible. 
- Split data into training and testing sets below.
- Convert all categorical data to one-hot vectors below
- Normalize all non-categorical data 
-  Perform KNN regression using a variety of values for n_neighbors (K) between 1 and 10 and both "uniform" and "distance" weights via a grid search where  *housing_labels* is the output and all other features are the input (similar to as seen in lecture two.)

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor

# =====================================================================
# Load Dataset
# =====================================================================
housing = pd.read_csv(os.path.join(".", "housing.csv"))

# =====================================================================
#  Stratified Sampling and split data (80% Train, 20% Test)
# =====================================================================
# Divide by 1.5 to limit the number of categories, and round up using ceil
housing["income_cat"] = np.ceil(housing["median_income"] / 1.5)
# Cap the categories above 5 at 5.0
housing["income_cat"] = housing["income_cat"].where(housing["income_cat"] < 5, 5.0)


# Perform 80/20 Stratified Shuffle Split
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in splitter.split(housing, housing["income_cat"]):
    strat_train_set = housing.iloc[train_index]
    strat_test_set = housing.iloc[test_index]

# Drop the auxiliary 'income_cat' column so the datasets return to their original schema
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

# Separate the training features (X_train) and the target labels (y_train)
housing_labels = strat_train_set["median_house_value"].copy()
housing_features = strat_train_set.drop("median_house_value", axis=1)

# Isolate feature attribute classes for the pipeline
housing_num = housing_features.drop("ocean_proximity", axis=1)
num_attribs = list(housing_num)
cat_attribs = ["ocean_proximity"]

# =====================================================================
# Custom Transformer Definition
# =====================================================================
rooms_ix, bedrooms_ix, population_ix, household_ix = 3, 4, 5, 6

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room=True): 
        self.add_bedrooms_per_room = add_bedrooms_per_room
        
    def fit(self, X, y=None):
        return self  # Structural transform only
        
    def transform(self, X, y=None):
        rooms_per_household = X[:, rooms_ix] / X[:, household_ix]
        population_per_household = X[:, population_ix] / X[:, household_ix]
        
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household, bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]

# =====================================================================
# Data Preprocessing Pipelines (with MinMaxScaler Normalization)
# =====================================================================
# Process numerical values: Impute -> add extra features -> Normalize to (-1, 1) range
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('attribs_adder', CombinedAttributesAdder()),
    ('min_max_scaler', MinMaxScaler(feature_range=(-1, 1)))
])

# Complete horizontal column pipeline mapping both numerical and categorical streams
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(), cat_attribs),
])

# Fit the parameters on the training subset features and transform them
housing_prepared = full_pipeline.fit_transform(housing_features)

# =====================================================================
#  Reconstruct NumPy Array into a clean Pandas DataFrame
# =====================================================================
extra_attribs = ["rooms_per_household", "population_per_household", "bedrooms_per_room"]
cat_encoder = full_pipeline.named_transformers_["cat"]
cat_one_hot_attribs = list(cat_encoder.categories_[0])

attributes = num_attribs + extra_attribs + cat_one_hot_attribs
housing_prepared_df = pd.DataFrame(
    housing_prepared,
    columns=attributes,
    index=housing_features.index
)

# =====================================================================
#  Hyperparameter Tuning with Grid Search & Cross-Validation
# =====================================================================
# Define search grid parameters: K choices 1 to 10 across both weight functions
param_grid = [
    {
        'n_neighbors': list(range(1, 11)),
        'weights': ['uniform', 'distance']
    }
]

# Set up Grid Search bound to 5-fold cross-validation over the prepared array
grid_search = GridSearchCV(
    estimator=KNeighborsRegressor(), 
    param_grid=param_grid, 
    cv=5, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1
)

# Run the parameter validation checks
grid_search.fit(housing_prepared, housing_labels)

# =====================================================================
#  Display Optimized Evaluation Outputs
# =====================================================================
print("Best Value Discovered:", grid_search.best_params_)

print("Best Estimator:", grid_search.best_estimator_)

best_rmse = np.sqrt(-grid_search.best_score_)
print(f"Best Cross-Validated RMSE: ${best_rmse:,.2f}")

Best Value Discovered: {'n_neighbors': 9, 'weights': 'distance'}
Best Estimator: KNeighborsRegressor(n_neighbors=9, weights='distance')
Best Cross-Validated RMSE: $59,915.15


### Conclusions
For what values of n_neighbors and weight does KNeighborsRegressor perform the best? Does it perform as well on the housing data as the linear regressor from the lectures? Why do you think this is?

<pre> WRITE RESPONSE HERE </pre>

The `KNeighborsRegressor` performs best with the following hyperparameter combination discovered through Grid Search:

- **n_neighbors (K):** 9  
- **weights:** `'distance'`

This model performed better than the Linear Regression model from the lecture, which had an RMSE of **68,321**.

---

## Why KNN Performs Better

Real estate prices are highly location-sensitive and usually do not follow a simple linear relationship.

### Linear Regression
Linear Regression assumes housing prices change uniformly across the map, similar to a flat plane. This assumption is often too rigid for real-world housing markets.

### KNN Regression
KNN focuses only on the closest neighboring houses and ignores distant regions. By averaging nearby house prices, it captures local patterns more effectively, such as:

- expensive coastal neighborhoods
- urban vs rural differences
- local income effects

Because of this, KNN can model complex regional pricing patterns that Linear Regression may miss.

### Read appending B

- Reflect on your last data project, read appendix B. Then, write down a few of the checklist items that your last data project could have used. If you have not yet done a data project, then write down a few of the items that you found most interesting.


## Reflection on My Data Project

Several checklist items were useful for my housing price prediction project.

- Understanding the problem before selecting a model
- Properly preprocessing the data
- Avoiding data leakage between training and test sets
- Using feature scaling (min_max normalization)
- Performing hyperparameter tuning with Grid Search(GridSearchCV)

These steps helped improve the performance and reliability of the model.

### Submit your notebook

Submit your solution to Quercus
Make sure you rename your notebook to    
W2_UTORid.ipynb    
Example W2_adfasd01.ipynb
